# 多Agent 完整示例

来源:
- https://docs.langchain.com/mcp/multi-agent/handoffs-customer-support
- https://docs.langchain.com/mcp/multi-agent/router-knowledge-base
- https://docs.langchain.com/mcp/multi-agent/skills-sql-assistant
- https://docs.langchain.com/mcp/multi-agent/subagents-personal-assistant

本笔记展示4个完整的多Agent应用示例：客服交接系统、知识库路由器、SQL助手(技能模式)、个人助理(子Agent模式)。

---
## 1. 客服交接系统 (Handoffs Customer Support)

**场景**: 一个通用客服Agent将复杂问题交接给专业Agent处理。

### 系统架构

```
[用户] → [前台Agent] → 判断问题类型
    ├── 账单问题 → [账单Agent]
    ├── 技术问题 → [技术Agent]
    ├── 退款问题 → [退款Agent]
    └── 简单问题 → 直接回答
```

### 代码实现

```python
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command
from typing import Literal

# 前台Agent - 第一线客服
def triage_agent(state: MessagesState) -> Command[Literal["billing", "tech", "refund", "__end__"]]:
    last_msg = state["messages"][-1].content
    
    # 使用LLM判断意图并路由
    classification = llm.with_structured_output(RouteDecision).invoke([
        ("system", "判断用户意图: billing(账单), tech(技术), refund(退款), general(通用)"),
        ("human", last_msg)
    ])
    
    if classification.route == "billing":
        return Command(goto="agent_billing", update={"messages": state["messages"]})
    elif classification.route == "tech":
        return Command(goto="agent_tech", update={"messages": state["messages"]})
    elif classification.route == "refund":
        return Command(goto="agent_refund", update={"messages": state["messages"]})
    else:
        # 直接回答通用问题
        response = llm.invoke(state["messages"])
        return Command(goto=END, update={"messages": [response]})

# 专业Agent
def billing_agent(state: MessagesState) -> Command[Literal["triage_agent", "__end__"]]:
    """账单专用Agent"""
    context = f"用户资料: {get_user_billing(state)}"
    response = llm.invoke([
        ("system", f"你是账单专家。{context} 如果问题超出范围，交回前台。"),
        *[(m.type, m.content) for m in state["messages"]],
    ])
    if "需要转回前台" in response.content:
        return Command(goto="triage_agent", update={"messages": [response]})
    return Command(goto=END, update={"messages": [response]})

def tech_agent(state: MessagesState) -> Command[Literal["triage_agent", "__end__"]]:
    """技术支持专用Agent"""
    response = llm.invoke([
        ("system", "你是高级技术支持工程师。提供详细的技术解决方案。"),
        *[(m.type, m.content) for m in state["messages"]],
    ])
    return Command(goto=END, update={"messages": [response]})

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("triage_agent", triage_agent)
builder.add_node("agent_billing", billing_agent)
builder.add_node("agent_tech", tech_agent)
builder.add_node("agent_refund", refund_agent)
builder.add_edge(START, "triage_agent")
graph = builder.compile()

# 运行示例
result = graph.invoke({"messages": [("human", "我上个月被多收了$50，能帮我查一下吗？")]})
```

---
## 2. 知识库路由器 (Router Knowledge Base)

**场景**: 一个路由Agent根据查询将请求分发到不同的知识库(文档、代码、FAQ)。

### 系统架构

```
[用户查询]
    │
    ▼
[Router Agent] ──→ 分类查询类型
    │
    ├── 产品文档 → [文档知识库 Agent] → 检索文档 + 回答
    ├── 代码示例 → [代码知识库 Agent] → 搜索代码 + 解释
    ├── 常见问题 → [FAQ Agent] → 匹配FAQ + 回答
    └── 通用问题 → [通用 Agent] → LLM直接回答
```

### 代码实现

```python
from pydantic import BaseModel, Field
from typing import Literal

class RouterOutput(BaseModel):
    """路由决策"""
    target: Literal["docs", "code", "faq", "general"] = Field(
        description="路由目标"
    )
    query: str = Field(description="优化后的查询")
    confidence: float = Field(ge=0, le=1, description="置信度")


def router_node(state: MessagesState) -> Command[Literal["doc_agent", "code_agent", "faq_agent", "general_agent"]]:
    """路由节点"""
    last_msg = state["messages"][-1].content
    
    route = llm.with_structured_output(RouterOutput).invoke([
        ("system", "将用户问题路由到最合适的知识库。"),
        ("human", last_msg)
    ])
    
    goto_map = {
        "docs": "doc_agent",
        "code": "code_agent",
        "faq": "faq_agent",
        "general": "general_agent",
    }
    return Command(goto=goto_map[route.target], update={"messages": state["messages"]})


def doc_agent(state: MessagesState) -> Command[Literal["__end__"]]:
    """文档知识库Agent"""
    query = state["messages"][-1].content
    docs = vector_store.similarity_search(query, k=5)
    context = "\n\n".join([d.page_content for d in docs])
    response = llm.invoke([
        ("system", f"基于以下文档回答问题:\n{context}"),
        ("human", query)
    ])
    return Command(goto=END, update={"messages": [response]})


def code_agent(state: MessagesState) -> Command[Literal["__end__"]]:
    """代码知识库Agent"""
    query = state["messages"][-1].content
    code_results = code_search(query)
    response = llm.invoke([
        ("system", f"基于以下代码示例回答:\n{code_results}"),
        ("human", query)
    ])
    return Command(goto=END, update={"messages": [response]})


# 构建图
builder = StateGraph(MessagesState)
builder.add_node("router_node", router_node)
builder.add_node("doc_agent", doc_agent)
builder.add_node("code_agent", code_agent)
builder.add_node("faq_agent", faq_agent)
builder.add_node("general_agent", general_agent)
builder.add_edge(START, "router_node")
graph = builder.compile()
```

---
## 3. SQL助手 (Skills SQL Assistant)

**场景**: 使用技能(Skills)模式构建的SQL数据库查询助手。

### 系统架构

```
[用户自然语言问题]
    │
    ▼
[SQL Agent]
    ├── 技能: schema_lookup (查表结构)
    ├── 技能: sql_generator (生成SQL)
    ├── 技能: query_executor (执行查询)
    └── 技能: result_analyzer (分析结果)
    │
    ▼
[自然语言回答]
```

### 代码实现

```python
from typing import List, Callable

# 定义技能
class Skill:
    def __init__(self, name: str, description: str, tools: List[Callable], prompt: str):
        self.name = name
        self.description = description
        self.tools = tools
        self.prompt = prompt

# 数据库模式查询技能
def get_table_schema(table_name: str) -> str:
    """获取数据库表结构"""
    return db.execute(f"SELECT sql FROM sqlite_master WHERE name='{table_name}';").fetchone()[0]

# SQL生成技能
def validate_sql(sql: str) -> bool:
    """验证SQL语法"""
    try:
        db.execute(f"EXPLAIN {sql}")
        return True
    except:
        return False

# 查询执行技能
def run_query(sql: str) -> str:
    """执行SQL查询"""
    result = db.execute(sql).fetchall()
    return str(result)

# 装配技能
sql_skills = [
    Skill("schema_lookup", "查询数据库表结构", [get_table_schema],
          "你可以查询数据库表结构来了解数据模型。"),
    Skill("sql_generator", "生成并验证SQL", [validate_sql],
          "你可以生成并验证SQL查询语句。"),
    Skill("query_executor", "执行查询", [run_query],
          "你可以安全执行SQL查询语句。"),
    Skill("result_analyzer", "分析结果", [],
          "你可以分析查询结果并以自然语言回答用户。"),
]

# 动态创建Agent
all_tools = [t for s in sql_skills for t in s.tools]
skill_prompts = "\n".join([s.prompt for s in sql_skills])

sql_agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=f"你是一个SQL数据库助手。{skill_prompts}\n"
            "步骤: 1.查表结构 2.生成SQL 3.验证 4.执行 5.分析结果"
)

# 使用示例
result = sql_agent.invoke({
    "messages": [("human", "查询上个月销售额最高的3个产品")]
})
print(result["messages"][-1].content)
```

---
## 4. 个人助理 (Subagents Personal Assistant)

**场景**: 一个个人助理Agent使用子Agent处理日程、邮件、任务等不同领域。

### 系统架构

```
[用户]
    │
    ▼
[个人助理 (Orchestrator)]
    │
    ├── [日历 Agent] ── 管理日程
    │      ├── 创建事件
    │      ├── 查询日程
    │      └── 冲突检测
    │
    ├── [邮件 Agent] ── 邮件管理
    │      ├── 发送邮件
    │      ├── 整理收件箱
    │      └── 邮件摘要
    │
    ├── [任务 Agent] ── 任务管理
    │      ├── 创建任务
    │      ├── 设置优先级
    │      └── 进度追踪
    │
    └── [搜索 Agent] ── 信息检索
           ├── 网络搜索
           └── 知识检索
```

### 代码实现

```python
from langgraph.prebuilt import create_react_agent

# 子Agent工厂函数
def create_calendar_agent():
    """创建日历子Agent"""
    return create_react_agent(
        model=llm,
        tools=[create_event, list_events, check_conflicts],
        prompt="你是日历管理助手。帮助用户管理日程。"
    )

def create_email_agent():
    """创建邮件子Agent"""
    return create_react_agent(
        model=llm,
        tools=[send_email, list_inbox, search_emails, summarize_thread],
        prompt="你是邮件管理助手。帮助用户处理邮件。"
    )

def create_task_agent():
    """创建任务子Agent"""
    return create_react_agent(
        model=llm,
        tools=[create_task, update_task, list_tasks, set_priority],
        prompt="你是任务管理助手。帮助用户管理待办事项。"
    )

# 主Orchestrator Agent
calendar_agent = create_calendar_agent()
email_agent = create_email_agent()
task_agent = create_task_agent()

def orchestrator(state: PersonalState) -> Command[Literal["calendar", "email", "task", "__end__"]]:
    """主协调器：理解用户意图并委派给子Agent"""
    last_msg = state["messages"][-1].content
    
    intent = llm.with_structured_output(IntentOutput).invoke([
        ("system", "判断用户意图: calendar(日历), email(邮件), task(任务)"),
        ("human", last_msg)
    ])
    
    result = {"messages": state["messages"], "context": state.get("context", {})}
    goto_map = {
        "calendar": "calendar_agent",
        "email": "email_agent",
        "task": "task_agent",
    }
    return Command(goto=goto_map.get(intent.domain, "calendar"))


def calendar_node(state: PersonalState) -> Command[Literal["orchestrator", "__end__"]]:
    """日历子Agent节点"""
    result = calendar_agent.invoke(state)
    if "需要协调" in str(result):
        return Command(goto="orchestrator", update=result)
    return Command(goto=END, update=result)


def email_node(state: PersonalState) -> Command[Literal["orchestrator", "__end__"]]:
    """邮件子Agent节点"""
    result = email_agent.invoke(state)
    if "需要协调" in str(result):
        return Command(goto="orchestrator", update=result)
    return Command(goto=END, update=result)


def task_node(state: PersonalState) -> Command[Literal["orchestrator", "__end__"]]:
    """任务子Agent节点"""
    result = task_agent.invoke(state)
    if "需要协调" in str(result):
        return Command(goto="orchestrator", update=result)
    return Command(goto=END, update=result)


# 构建图
builder = StateGraph(PersonalState)
builder.add_node("orchestrator", orchestrator)
builder.add_node("calendar_agent", calendar_node)
builder.add_node("email_agent", email_node)
builder.add_node("task_agent", task_node)
builder.add_edge(START, "orchestrator")
graph = builder.compile()

# 运行示例
result = graph.invoke({
    "messages": [("human", "明天下午3点安排一个团队会议，然后发邮件给团队通知")],
    "context": {}
})
```

---
## 5. 对比总结

| 示例 | 核心模式 | 适用场景 | 优势 |
|------|----------|----------|------|
| 客服交接系统 | Handoffs | 客服、技术支持 | 专业化处理，平滑转接 |
| 知识库路由器 | Router | 多知识库、多领域 | 集中路由，精准分发 |
| SQL助手 | Skills | 数据库查询、数据分析 | 模块化能力扩展 |
| 个人助理 | Subagents | 个人助手、办公自动化 | 职责清晰，可扩展 |

## 6. 参考链接

- [Handoffs 客服示例](https://docs.langchain.com/mcp/multi-agent/handoffs-customer-support)
- [Router 知识库示例](https://docs.langchain.com/mcp/multi-agent/router-knowledge-base)
- [Skills SQL助手示例](https://docs.langchain.com/mcp/multi-agent/skills-sql-assistant)
- [Subagents 个人助理示例](https://docs.langchain.com/mcp/multi-agent/subagents-personal-assistant)
- [LangGraph 文档](https://langchain-ai.github.io/langgraph/)
- [LangGraph GitHub](https://github.com/langchain-ai/langgraph)